In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error

# Load the dataset
df = pd.read_csv('data.csv')

# Ensure target column is named correctly based on your specific task (e.g., 'fuel_efficiency_mpg')
# Replace 'fuel_efficiency_mpg' with your actual target variable name if it differs
target_col = 'fuel_efficiency_mpg'

print("--- Question 1: Missing Values ---")
for column in df.columns:
    missing_count = df[column].isnull().sum()
    if missing_count > 0:
        print(f"Column '{column}' has {missing_count} missing values.")


# Question 2: Median for horsepower

print("--- Question 2: Median for Horsepower ---")
median_hp = df['Engine HP'].median()
print(f"Median for horsepower: {median_hp}\n")

# ==========================================
# Helper Function for RMSE Calculation
# ==========================================
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# Prepare features (excluding target)
features = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']

# ==========================================
# Question 3: Filling NAs (With 0 vs With Mean)
# ==========================================
print("--- Question 3: Filling NAs ---")
for fill_value in [0, df['Engine HP'].mean()]:
    df_filled = df.copy()
    df_filled['Engine HP'] = df_filled['Engine HP'].fillna(fill_value)
    
    # Split: 60% Train, 20% Val, 20% Test (Seed 42)
    df_train_val, df_test = train_test_split(df_filled, test_size=0.2, random_state=42)
    df_train, df_val = train_test_split(df_train_val, test_size=0.25, random_state=42)
    
    X_train, y_train = df_train[features], df_train[target_col]
    X_val, y_val = df_val[features], df_val[target_col]
    
    model = LinearRegression()
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    
    fill_type = "0" if fill_value == 0 else "mean"
    print(f"Validation RMSE when filling with {fill_type}: {round(rmse(y_val, preds), 2)}")
print()

# For the subsequent questions, fill NAs with 0 as standard protocol
df['Engine HP'] = df['Engine HP'].fillna(0)

# ==========================================
# Question 4: Best Regularization (Ridge)
# ==========================================
print("--- Question 4: Best Regularization ---")
df_train_val, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_train_val, test_size=0.25, random_state=42)

X_train, y_train = df_train[features], df_train[target_col]
X_val, y_val = df_val[features], df_val[target_col]

alphas = [0, 0.01, 0.1, 1, 5, 10, 100]
for a in alphas:
    ridge = Ridge(alpha=a, random_state=42)
    ridge.fit(X_train, y_train)
    preds = ridge.predict(X_val)
    print(f"Alpha: {a} -> RMSE: {round(rmse(y_val, preds), 2)}")
print()

# ==========================================
# Question 5: RMSE Standard Deviation (Multiple Seeds)
# ==========================================
print("--- Question 5: RMSE Standard Deviation ---")
seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
rmse_scores = []

for seed in seeds:
    df_train_val, df_test = train_test_split(df, test_size=0.2, random_state=seed)
    df_train, df_val = train_test_split(df_train_val, test_size=0.25, random_state=seed)
    
    X_train, y_train = df_train[features], df_train[target_col]
    X_val, y_val = df_val[features], df_val[target_col]
    
    model = LinearRegression()
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    rmse_scores.append(rmse(y_val, preds))

std_deviation = np.std(rmse_scores)
print(f"RMSE scores across seeds: {[round(r, 4) for r in rmse_scores]}")
print(f"Standard Deviation: {round(std_deviation, 3)}\n")

# ==========================================
# Question 6: Evaluation on Test Set
# ==========================================
print("--- Question 6: Evaluation on Test ---")
# Split with seed 9
df_train_val, df_test = train_test_split(df, test_size=0.2, random_state=9)
df_train, df_val = train_test_split(df_train_val, test_size=0.25, random_state=9)

# Combine Train and Validation datasets as instructed for final test evaluation
df_full_train = pd.concat([df_train, df_val])

X_full_train, y_full_train = df_full_train[features], df_full_train[target_col]
X_test, y_test = df_test[features], df_test[target_col]

# Train with best regularized model (e.g., alpha=0.001 or regular linear regression depending on instructions)
final_model = LinearRegression()
final_model.fit(X_full_train, y_full_train)
test_preds = final_model.predict(X_test)

print(f"Final Test RMSE: {round(rmse(y_test, test_preds), 3)}")

--- Question 1: Missing Values ---
Column 'Engine Fuel Type' has 3 missing values.
Column 'Engine HP' has 69 missing values.
Column 'Engine Cylinders' has 30 missing values.
Column 'Number of Doors' has 6 missing values.
Column 'Market Category' has 3742 missing values.
--- Question 2: Median for Horsepower ---
Median for horsepower: 227.0

--- Question 3: Filling NAs ---


KeyError: "None of [Index(['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year'], dtype='str')] are in the [columns]"